# Notebook 01 – Data Cleaning & Preprocessing

**Evidence-Based Adaptive User Interface Framework for E-Commerce**

### Objectives
- Load raw survey data
- Validate integrity
- Remove invalid responses
- Verify attention checkers (`check1`–`check5`): keep if **≥ 4/5 correct** (at most 1 wrong); drop if **≥ 2 wrong** (likely low-effort)
- Handle missing values
- Standardize column names
- Export clean dataset


## Expected Outputs
- `data/processed/clean_dataset.csv`
- `reports/01_data_cleaning/data_quality_summary.xlsx`
- `reports/01_data_cleaning/data_cleaning_report.md`


In [12]:
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)

    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate

    raise FileNotFoundError(
        "Could not find project root containing src/config.py. "
        "Open EvidenceBasedAdaptiveUI (or its parent workspace) and rerun."
    )


_bootstrap_project()
from src.preprocessing.cleaning import (
    add_big_five_levels,
    categorical_quality_report,
    compute_big_five_scores,
    dataset_summary,
    drop_bfi10_raw_columns,
    filter_attention_checks,
    score_attention_checks,
    standardize_column_names,
)
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

PATHS, REPORTS = setup_notebook("01_data_cleaning")
RAW_DATA = PATHS.data_raw / "E-Commerce - data.csv"
PROCESSED = PATHS.data_processed


## Load Dataset


In [13]:
df = pd.read_csv(RAW_DATA)
print(df.shape)
display(df.head())
df.columns


(240, 74)


,Timestamp,consent,age_group,gender,education_level,primary_device,shopping_motivation,decision_speed,price_sensitivity,review_importance,...,mobile_price_display,mobile_filter_location,mobile_quick_view,mobile_review_display,mobile_sticky_header,mobile_touch_size,expects_device_adaptation,adaptation_comfort,delight_factors,frustration_points
0,20/01/2026 17:24:30,Yes,18-24,Male,Graduate,Smartphone,Research (I like exploring products and learni...,Quick (5-15 minutes - I know what I want),Moderately important (I balance price and qual...,Sometimes read reviews (For certain product ty...,...,Strike-through Original (Shows original price ...,Drawer Overlay (Slides in from the side when c...,Slide-Up Panel (Preview product details in a s...,Collapsible Sections (Reviews can be expanded ...,No (Header scrolls away with content),"Large (Easier to tap, less content visible)",Somewhat Different (Adapted but recognizable),3,NaN,NaN
1,20/01/2026 17:27:08,Yes,25-34,Female,Graduate,Smartphone,Deal-hunting (I look for sales and discounts),Moderate (15-30 minutes - I compare a few opti...,Moderately important (I balance price and qual...,Often read reviews (For most purchases),...,Strike-through Original (Shows original price ...,Drawer Overlay (Slides in from the side when c...,Slide-Up Panel (Preview product details in a s...,Collapsible Sections (Reviews can be expanded ...,Yes (Header stays at top while scrolling),"Standard (Normal size, more content visible)",Somewhat Different (Adapted but recognizable),5,the website or the application should be user-...,when the description or the photo of the produ...
2,20/01/2026 17:37:31,Yes,25-34,Female,Graduate,Smartphone,Gift-buying (I mainly shop for others),Moderate (15-30 minutes - I compare a few opti...,Moderately important (I balance price and qual...,Always read reviews (I won't buy without check...,...,Strike-through Original (Shows original price ...,Drawer Overlay (Slides in from the side when c...,Slide-Up Panel (Preview product details in a s...,Collapsible Sections (Reviews can be expanded ...,Yes (Header stays at top while scrolling),"Standard (Normal size, more content visible)","Yes, Very Different (Completely different desi...",5,NaN,NaN
3,20/01/2026 17:38:38,Yes,18-24,Male,Graduate,Smartphone,Entertainment (I browse for fun and enjoyment),Impulsive (Less than 5 minutes - I decide quic...,Somewhat unimportant (Price matters but isn't ...,Sometimes read reviews (For certain product ty...,...,Simple Display (Price is visible but not empha...,Drawer Overlay (Slides in from the side when c...,Slide-Up Panel (Preview product details in a s...,Summary Only (Average rating and total count),Yes (Header stays at top while scrolling),"Large (Easier to tap, less content visible)",Somewhat Different (Adapted but recognizable),4,It allows me to buy items new to me and it sav...,Lack of stock or high prices for low quality
4,20/01/2026 17:41:19,Yes,25-34,Female,Postgraduate,Smartphone,Need-based (I shop when I need something speci...,Research-heavy (30+ minutes - I thoroughly res...,Moderately important (I balance price and qual...,Often read reviews (For most purchases),...,Simple Display (Price is visible but not empha...,Drawer Overlay (Slides in from the side when c...,No Quick View (I prefer to go to the full prod...,Collapsible Sections (Reviews can be expanded ...,Yes (Header stays at top while scrolling),"Large (Easier to tap, less content visible)","Yes, Very Different (Completely different desi...",5,Search bar - filters - colors of the pages,If I can’t sea the prices clearly or no filter...


Index(['Timestamp', 'consent', 'age_group', 'gender', 'education_level',
       'primary_device', 'shopping_motivation', 'decision_speed',
       'price_sensitivity', 'review_importance', 'brand_loyalty',
       'social_proof_influence', 'primary_persona', 'check1',
       'trait_introversion', 'trait_trust', 'trait_low_conscientiousness',
       'trait_emotional_stability', 'check2', 'trait_low_openness',
       'trait_extraversion', 'trait_agreeableness_reverse',
       'trait_conscientiousness', 'trait_neuroticism', 'trait_openness',
       'current_mood', 'font_style_pref', 'font_size_pref', 'color_theme_pref',
       'accent_color_pref', 'background_pref', 'whitespace_pref',
       'button_style_pref', 'hero_banner_size', 'recommendation_type',
       'social_proof_display', 'urgency_pref', 'check3', 'checkout_style',
       'form_field_style', 'product_desc_length', 'check4',
       'desktop_grid_pref', 'desktop_info_density', 'desktop_image_text_ratio',
       'desktop_whitespac

## Dataset Quality


In [14]:
summary = dataset_summary(df)
display(summary)
summary.to_excel(REPORTS / "data_quality_summary.xlsx")


,dtype,missing,unique
Timestamp,object,0,240
consent,object,0,1
age_group,object,0,4
gender,object,0,2
education_level,object,0,5
...,...,...,...
mobile_touch_size,object,0,3
expects_device_adaptation,object,0,4
adaptation_comfort,int64,0,5
delight_factors,object,88,149


## Attention Check (`check1`–`check5`)

Embedded attention items catch rushed / “for fun” answering.

**Rule used here**
- Correct answers: `check1=3`, `check2=3`, `check3="Rounded Corners"`, `check4=3`, `check5=3`
- **Keep** a row if at least **4 out of 5** checkers are correct (≤ 1 wrong)
- **Remove** a row if **more than 1** checker is wrong (≥ 2 wrong) — treated as misleading

We print counts **before** filtering, the wrong-count distribution, then how many were removed / kept.


In [15]:
from src.preprocessing.cleaning import score_attention_checks

# --- Counts BEFORE removing any attention-fail rows ---
before_attention = len(df)
print(f"Records before attention filter: {before_attention}")

scored = score_attention_checks(df)
wrong_dist = (
    scored["attention_n_wrong"]
    .value_counts()
    .sort_index()
    .rename_axis("n_wrong_checkers")
    .reset_index(name="n_records")
)
wrong_dist["pct"] = (100 * wrong_dist["n_records"] / before_attention).round(1)
print("\nDistribution of wrong attention checkers (before removal):")
display(wrong_dist)

n_keep_4of5 = int((scored["attention_n_correct"] >= 4).sum())
n_drop_2plus = int((scored["attention_n_wrong"] >= 2).sum())
n_exactly_1_wrong = int((scored["attention_n_wrong"] == 1).sum())
n_all_correct = int((scored["attention_n_wrong"] == 0).sum())

print(f"\nAll 5 correct:              {n_all_correct}")
print(f"Exactly 1 wrong (KEEP):     {n_exactly_1_wrong}")
print(f"Would KEEP (>=4/5 correct): {n_keep_4of5}")
print(f"Would REMOVE (>=2 wrong):   {n_drop_2plus}")

# Apply filter: keep if >= 4/5 correct (max 1 wrong)
df = filter_attention_checks(df, min_correct=4)

print(f"\nRemoved {before_attention - len(df)} responses with >= 2 wrong checkers")
print(f"Remaining rows after attention filter: {len(df)}")


Records before attention filter: 240

Distribution of wrong attention checkers (before removal):


,n_wrong_checkers,n_records,pct
0,0,200,83.3
1,1,24,10.0
2,2,2,0.8
3,3,6,2.5
4,4,5,2.1
5,5,3,1.2


INFO: Attention checkers before filter: 240 rows | wrong-count distribution: {0: 200, 1: 24, 2: 2, 3: 6, 4: 5, 5: 3}
INFO: Keep rule: >= 4/5 correct (max 1 wrong). Kept 224, removed 16



All 5 correct:              200
Exactly 1 wrong (KEEP):     24
Would KEEP (>=4/5 correct): 224
Would REMOVE (>=2 wrong):   16

Removed 16 responses with >= 2 wrong checkers
Remaining rows after attention filter: 224


In [16]:
# Optional: compare old strict rule (all 5 must be correct) vs new 4/5 rule
# (recompute on a fresh score of the pre-filter snapshot is not available here;
#  the numbers printed above already show how many 1-wrong rows we keep.)
print("Policy summary")
print("  Old policy: require 5/5 correct")
print("  New policy: require >= 4/5 correct (tolerate 1 mistake)")
print("  Rationale: one slip ≠ low-effort; 2+ wrong checkers ≈ unreliable answers")


Policy summary
  Old policy: require 5/5 correct
  New policy: require >= 4/5 correct (tolerate 1 mistake)
  Rationale: one slip ≠ low-effort; 2+ wrong checkers ≈ unreliable answers


## Duplicate Check


In [17]:
if "timestamp" in df.columns:
    duplicates = df["timestamp"].duplicated().sum()
    print(f"Duplicate timestamps: {duplicates}")
else:
    print("Timestamp column not found")


Timestamp column not found


## Missing Values


In [18]:
display(df.isna().sum().sort_values(ascending=False))


frustration_points             81
delight_factors                79
desktop_navigation              0
desktop_persistent_filters      0
desktop_filter_location         0
                               ..
trait_openness                  0
trait_neuroticism               0
trait_conscientiousness         0
trait_agreeableness_reverse     0
check3                          0
Length: 74, dtype: int64

## Standardize Columns


In [19]:
df = standardize_column_names(df)
print(df.columns.tolist())


['timestamp', 'consent', 'age_group', 'gender', 'education_level', 'primary_device', 'shopping_motivation', 'decision_speed', 'price_sensitivity', 'review_importance', 'brand_loyalty', 'social_proof_influence', 'primary_persona', 'check1', 'trait_introversion', 'trait_trust', 'trait_low_conscientiousness', 'trait_emotional_stability', 'check2', 'trait_low_openness', 'trait_extraversion', 'trait_agreeableness_reverse', 'trait_conscientiousness', 'trait_neuroticism', 'trait_openness', 'current_mood', 'font_style_pref', 'font_size_pref', 'color_theme_pref', 'accent_color_pref', 'background_pref', 'whitespace_pref', 'button_style_pref', 'hero_banner_size', 'recommendation_type', 'social_proof_display', 'urgency_pref', 'check3', 'checkout_style', 'form_field_style', 'product_desc_length', 'check4', 'desktop_grid_pref', 'desktop_info_density', 'desktop_image_text_ratio', 'desktop_whitespace', 'desktop_navigation', 'desktop_search_visibility', 'desktop_category_display', 'check5', 'desktop_pr

## Categorical Validation


In [20]:
quality_df = categorical_quality_report(df)
display(quality_df)
quality_df.to_excel(REPORTS / "categorical_quality_summary.xlsx", index=False)


,column,unique_values,missing
0,timestamp,224,0
1,consent,1,0
2,age_group,4,0
3,gender,2,0
4,education_level,5,0
5,primary_device,2,0
6,shopping_motivation,5,0
7,decision_speed,4,0
8,price_sensitivity,5,0
9,review_importance,5,0


## Big Five Scores


In [21]:
df = compute_big_five_scores(df)
df = add_big_five_levels(df)
df = drop_bfi10_raw_columns(df)
print(f"Remaining columns: {df.shape[1]}")


Remaining columns: 79


## Export


In [22]:
output = PROCESSED / "clean_dataset.csv"
df.to_csv(output, index=False)

with open(REPORTS / "cleaning_log.txt", "w", encoding="utf-8") as handle:
    handle.write(f"Final rows: {len(df)}\n")
    handle.write(f"Columns: {df.shape[1]}\n")

with open(REPORTS / "data_cleaning_report.md", "w", encoding="utf-8") as handle:
    handle.write("# Data Cleaning Report\n\n")
    handle.write(f"- Final rows: {len(df)}\n")
    handle.write(f"- Final columns: {df.shape[1]}\n")
    handle.write(f"- Output file: `{output}`\n")

print(f"Saved: {output}")


Saved: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/processed/clean_dataset.csv
